In [ ]:
!pip install -q transformers accelerate bitsandbytes sentence-transformers rank_bm25 faiss-cpu joblib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 37.4 MB/s eta 0:00:00


In [ ]:
import joblib
import json
import random
from tqdm import tqdm
from google.colab import drive

drive.mount('/content/drive')
path = "/content/drive/MyDrive/Studia_MJ/LLM_projekt"

data = joblib.load(f'{path}/data_pack.joblib')
documents = data['documents']

Mounted at /content/drive


In [ ]:
def generate_test_set(model, tokenizer, documents, num_questions=30):
    test_set = []

    sampled_indices = random.sample(range(len(documents)), num_questions)

    print(f"Generting {num_questions} questions")

    for idx in tqdm(sampled_indices):
        doc_text = documents[idx]

        messages = [
            {
                "role": "system",
                "content": "You are a precise assistant that creates cosmetic ingredient datasets in JSON format."
            },
            {
                "role": "user",
                "content": f"Based on this cosmetic ingredient description, create a user question and a factual answer.\n\nDOCUMENT CONTENT:\n{doc_text}\n\nFormat your response as a JSON object: {{\"question\": \"...\", \"expected_answer\": \"...\"}}"
            }
        ]

        prompt_text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=300,
                temperature=0.7,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id
            )

        response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

        try:
            start = response.find('{')
            end = response.rfind('}') + 1
            test_item = json.loads(response[start:end])
            test_item['document_i'] = idx

            test_set.append(test_item)
        except:
            continue

    return test_set

In [ ]:
from huggingface_hub import login

login()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

model_id = "Qwen/Qwen2.5-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16
)


tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    low_cpu_mem_usage=True
)



config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

In [ ]:
my_test_set = generate_test_set(model, tokenizer, documents, num_questions=50)

print(my_test_set[0])

Generting 50 questions


100%|██████████| 50/50 [08:22<00:00, 10.04s/it]

{'question': 'What is the primary function of Methyl Gluceth-10 in cosmetic formulations?', 'expected_answer': 'Methyl Gluceth-10 works as a humectant ingredient helping the skin to cling onto water.', 'document_i': 880}


In [ ]:
with open(f'{path}/synthetic_test_set.json', 'w', encoding='utf-8') as f:
    json.dump(my_test_set, f, ensure_ascii=False, indent=4)

print(f"Path to test set {path}/synthetic_test_set.json")

Path to test set /content/drive/MyDrive/Studia_MJ/LLM_projekt/synthetic_test_set.json


In [ ]:
print(my_test_set[0])
print(documents[880])

print()

print(my_test_set[1])
print(documents[154])

{'question': 'What is the primary function of Methyl Gluceth-10 in cosmetic formulations?', 'expected_answer': 'Methyl Gluceth-10 works as a humectant ingredient helping the skin to cling onto water.', 'document_i': 880}
Ingredient: Methyl Gluceth-10 | Description: A pale yellow, corn-derived liquid that works as a humectant ingredient helping the skin to cling onto water. It has a smooth, silky feel and can reduce the tackiness of other humectants.

{'question': 'What is the source of Leuconostoc/Radish Root Ferment Filtrate and what makes it a good preservative in cosmetics?', 'expected_answer': 'Leuconostoc/Radish Root Ferment Filtrate is sourced from radishes fermented with Leuconostoc kimchii, a lactic acid bacteria. The antimicrobial peptide secreted during the fermentation process gives it significant preservative properties, making it a promising natural alternative for use in cosmetics, though its effectiveness is lower than common chemical preservatives like parabens or pheno